# Sistemas Lineares — Métodos Numéricos

**Engenharia Naval e Oceânica — Poli-USP**

Este notebook aborda a resolução numérica de sistemas lineares $A\mathbf{x} = \mathbf{b}$, tema central em Métodos Numéricos e com aplicações diretas em Engenharia Naval: análise estrutural de cascos por elementos finitos, resolução de malhas de escoamento (CFD), problemas de hidrostática e estabilidade, e otimização de projeto.

## Conteúdo

1. Introdução e motivação: métodos diretos vs. iterativos
2. Eliminação de Gauss com pivoteamento parcial
3. Método de Gauss-Seidel
4. Convergência de Gauss-Seidel (dominância diagonal e raio espectral)
5. Comparação final entre os métodos
6. Para ir além: Jacobi vs. Gauss-Seidel

---

In [ ]:
# Bibliotecas usadas ao longo do notebook.
# Observação: numpy.linalg.solve NÃO será usado para resolver sistemas — vamos
# implementar os algoritmos (Gauss, Gauss-Seidel, Jacobi) do zero. numpy será
# usado apenas para álgebra auxiliar (vetores, normas) e, na seção de
# convergência, para calcular autovalores com fins de ANÁLISE teórica.
import numpy as np
import matplotlib.pyplot as plt

# Paleta de cores consistente para todo o notebook
COR_GAUSS = '#1f77b4'      # azul - Eliminação de Gauss
COR_SEIDEL = '#2ca02c'     # verde - Gauss-Seidel
COR_JACOBI = '#d62728'     # vermelho - Jacobi
COR_DIVERGE = '#ff7f0e'    # laranja - caso divergente/instável
COR_DESTAQUE = '#9467bd'   # roxo - anotações/destaques

plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

np.set_printoptions(precision=6, suppress=True)
print('Ambiente pronto.')

## 1. Introdução e motivação

### Por que resolver $A\mathbf{x} = \mathbf{b}$ numericamente?

Sistemas lineares aparecem em praticamente toda a engenharia naval: ao discretizar a equação de Laplace para o escoamento potencial ao redor de um casco, ao montar a matriz de rigidez de um modelo de elementos finitos de uma viga-navio, ou ao resolver as equações de equilíbrio de um sistema de amarração com múltiplas linhas. Em todos esses casos, o problema contínuo é transformado em um sistema algébrico

$$A\mathbf{x} = \mathbf{b}, \qquad A \in \mathbb{R}^{n\times n}, \quad \mathbf{x}, \mathbf{b} \in \mathbb{R}^n$$

Quando $n$ é pequeno, poderíamos em tese calcular $\mathbf{x} = A^{-1}\mathbf{b}$ analiticamente (regra de Cramer, cofatores etc.). Mas isso é **inviável computacionalmente** para $n$ grande: o número de operações explode, e calcular a inversa explicitamente é desnecessário e numericamente arriscado. Precisamos de **algoritmos numéricos eficientes e estáveis**.

### Métodos diretos vs. métodos iterativos

Existem duas grandes famílias de métodos para resolver $A\mathbf{x}=\mathbf{b}$:

- **Métodos diretos**: obtêm a solução exata (a menos de erros de arredondamento) em um número *finito e predeterminado* de operações. O exemplo clássico é a **Eliminação de Gauss**. São robustos e previsíveis, mas o custo cresce rapidamente com $n$.

- **Métodos iterativos**: partem de uma estimativa inicial $\mathbf{x}^{(0)}$ e geram uma sequência $\mathbf{x}^{(1)}, \mathbf{x}^{(2)}, \ldots$ que **converge** para a solução. Exemplos: Jacobi e **Gauss-Seidel**. São vantajosos quando $A$ é grande e **esparsa** (muitos zeros), situação típica de malhas de elementos finitos/diferenças finitas em problemas navais, pois cada iteração custa pouco e não é preciso armazenar/fatorizar a matriz cheia.

### Custo computacional

A Eliminação de Gauss sobre uma matriz densa $n\times n$ requer aproximadamente

$$\frac{2n^3}{3} \text{ operações de ponto flutuante}$$

ou seja, custo $O(n^3)$. Isso é ótimo para sistemas pequenos/médios e densos, mas se torna proibitivo para $n$ da ordem de $10^5$–$10^6$ (comum em malhas 3D de FEM/CFD).

Já um método iterativo, aplicado a uma matriz esparsa com $\mathcal{O}(n)$ elementos não nulos, custa aproximadamente $O(n)$ **por iteração**. Se o método converge em poucas iterações (o que depende das propriedades de $A$), o custo total pode ser muito menor que $O(n^3)$. Essa é a razão prática pela qual, em simulações de grande escala, preferimos métodos iterativos (ou variantes mais sofisticadas, como gradientes conjugados e multigrid).

Nas próximas seções implementaremos, **do zero**, um representante de cada família: Eliminação de Gauss (direto) e Gauss-Seidel (iterativo), comparando-os ao final.

## 2. Eliminação de Gauss

### Ideia central

A Eliminação de Gauss resolve $A\mathbf{x} = \mathbf{b}$ trabalhando sobre a **matriz aumentada**

$$[A \mid \mathbf{b}] = \begin{bmatrix} a_{11} & a_{12} & \cdots & a_{1n} & b_1 \\ a_{21} & a_{22} & \cdots & a_{2n} & b_2 \\ \vdots & & \ddots & & \vdots \\ a_{n1} & a_{n2} & \cdots & a_{nn} & b_n \end{bmatrix}$$

O algoritmo tem duas fases:

1. **Escalonamento (eliminação progressiva)**: usando operações elementares de linha (trocar linhas, multiplicar uma linha por um escalar não nulo, somar um múltiplo de uma linha a outra — que não alteram o conjunto-solução), zeramos sistematicamente os elementos **abaixo da diagonal principal**, coluna por coluna, até obter uma matriz triangular superior:

$$\begin{bmatrix} u_{11} & u_{12} & \cdots & u_{1n} & c_1 \\ 0 & u_{22} & \cdots & u_{2n} & c_2 \\ \vdots & & \ddots & & \vdots \\ 0 & 0 & \cdots & u_{nn} & c_n \end{bmatrix}$$

   Para zerar o elemento $a_{ik}$ (linha $i$, coluna $k$, com $i>k$), subtraímos da linha $i$ um múltiplo da linha-pivô $k$:

   $$m_{ik} = \frac{a_{ik}}{a_{kk}}, \qquad \text{linha}_i \leftarrow \text{linha}_i - m_{ik}\cdot \text{linha}_k$$

   onde $a_{kk}$ é chamado **pivô**.

2. **Substituição regressiva (back-substitution)**: com a matriz já triangular superior, resolvemos de baixo para cima:

$$x_n = \frac{c_n}{u_{nn}}, \qquad x_i = \frac{1}{u_{ii}}\left(c_i - \sum_{j=i+1}^{n} u_{ij}\,x_j\right), \quad i = n-1, \ldots, 1$$

### Por que pivoteamento parcial?

Se o pivô $a_{kk}$ for **muito pequeno** (ou nulo), o multiplicador $m_{ik} = a_{ik}/a_{kk}$ fica enorme, amplificando qualquer erro de arredondamento presente nos outros elementos da linha — a operação linha$_i$ $-$ $m_{ik}\cdot$linha$_k$ passa a envolver a subtração de números de magnitudes muito diferentes, destruindo algarismos significativos (erro de cancelamento catastrófico).

O **pivoteamento parcial** evita isso: antes de eliminar a coluna $k$, procuramos entre as linhas $k, k+1, \ldots, n$ aquela com o **maior valor absoluto** na coluna $k$ e a trocamos com a linha $k$. Isso garante $|m_{ik}| \le 1$ sempre, controlando a propagação de erros. Vamos demonstrar isso numericamente a seguir.

In [ ]:
def eliminacao_gauss(A, b, pivoteamento=True, guardar_etapas=False):
    """Resolve A x = b por Eliminação de Gauss, implementada do zero.

    Parâmetros
    ----------
    A, b : matriz e vetor do sistema (não são modificados; trabalhamos em cópias).
    pivoteamento : se True, aplica pivoteamento parcial (troca de linhas pelo
        maior valor absoluto na coluna) antes de eliminar cada coluna.
    guardar_etapas : se True, guarda uma cópia da matriz aumentada após cada
        etapa de eliminação, para fins de visualização didática.

    Retorna
    -------
    x : vetor solução
    etapas : lista de matrizes aumentadas (uma por etapa), só se guardar_etapas=True
    """
    n = len(b)
    # Montamos a matriz aumentada [A | b] como array de ponto flutuante.
    M = np.hstack([A.astype(float), b.reshape(-1, 1).astype(float)])
    etapas = [M.copy()] if guardar_etapas else None

    # --- Fase 1: escalonamento (eliminação progressiva) ---
    for k in range(n - 1):
        if pivoteamento:
            # Procuramos, na coluna k, a linha (entre k e n-1) com maior |valor|.
            # Isso evita dividir por um pivô pequeno (ou nulo) na próxima etapa.
            linha_pivo = k + np.argmax(np.abs(M[k:, k]))
            if linha_pivo != k:
                M[[k, linha_pivo]] = M[[linha_pivo, k]]

        pivo = M[k, k]
        if pivo == 0.0:
            # Só barramos pivô EXATAMENTE nulo (sistema singular). Pivôs apenas
            # pequenos são deliberadamente permitidos para que possamos
            # demonstrar, a seguir, a instabilidade numérica que eles causam
            # quando o pivoteamento está desligado.
            raise ZeroDivisionError(f'Pivô nulo na coluna {k}: sistema singular (ou pivoteamento necessário).')

        # Zeramos todos os elementos abaixo do pivô na coluna k.
        for i in range(k + 1, n):
            m = M[i, k] / pivo  # multiplicador da linha i em relação à linha-pivô k
            M[i, k:] = M[i, k:] - m * M[k, k:]

        if guardar_etapas:
            etapas.append(M.copy())

    # --- Fase 2: substituição regressiva ---
    x = np.zeros(n)
    for i in range(n - 1, -1, -1):
        soma = np.dot(M[i, i + 1:n], x[i + 1:n])
        x[i] = (M[i, n] - soma) / M[i, i]

    if guardar_etapas:
        return x, etapas
    return x


# --- Teste rápido com um sistema 3x3 simples ---
A_teste = np.array([[2.0, 1.0, -1.0],
                     [-3.0, -1.0, 2.0],
                     [-2.0, 1.0, 2.0]])
b_teste = np.array([8.0, -11.0, -3.0])

x_teste = eliminacao_gauss(A_teste, b_teste)
print('Solução encontrada: x =', x_teste)
print('Resíduo ||A x - b|| =', np.linalg.norm(A_teste @ x_teste - b_teste))

### Visualizando o escalonamento

Para enxergar o que o algoritmo faz, vamos visualizar a matriz aumentada $[A\mid \mathbf{b}]$ como um "heatmap" (mapa de cores) antes e depois de cada etapa de eliminação. A cor representa a magnitude de cada elemento; a coluna mais à direita (separada por uma linha tracejada) é o vetor $\mathbf{b}$. Observe os zeros (região escura) sendo criados abaixo da diagonal, coluna a coluna.

In [ ]:
def plotar_etapas_eliminacao(etapas, titulo='Escalonamento da matriz aumentada [A | b]'):
    """Plota lado a lado a matriz aumentada em cada etapa da eliminação,
    como um heatmap, para visualizar os zeros sendo criados abaixo da diagonal.
    """
    n_etapas = len(etapas)
    fig, eixos = plt.subplots(1, n_etapas, figsize=(4.2 * n_etapas, 4))
    if n_etapas == 1:
        eixos = [eixos]

    # Usamos a mesma escala de cor (vmin/vmax) em todos os painéis para
    # comparação justa entre etapas.
    vmax = max(np.max(np.abs(M)) for M in etapas)

    for k, (ax, M) in enumerate(zip(eixos, etapas)):
        im = ax.imshow(np.abs(M), cmap='viridis', vmin=0, vmax=vmax, aspect='auto')
        n = M.shape[0]
        # Linha tracejada separando A de b
        ax.axvline(x=n - 0.5, color='white', linestyle='--', linewidth=1.5)
        ax.set_title(f'Etapa {k}' if k > 0 else 'Matriz original')
        ax.set_xticks(range(M.shape[1]))
        rotulos = [f'$a_{{\\cdot{j+1}}}$' for j in range(n)] + ['$b$']
        ax.set_xticklabels(rotulos)
        ax.set_yticks(range(n))
        ax.set_yticklabels([f'L{i+1}' for i in range(n)])
        # Anota os valores numéricos sobre cada célula
        for i in range(M.shape[0]):
            for j in range(M.shape[1]):
                cor_texto = 'white' if abs(M[i, j]) > vmax / 2 else 'black'
                ax.text(j, i, f'{M[i, j]:.1f}', ha='center', va='center', color=cor_texto, fontsize=9)

    fig.suptitle(titulo, fontsize=13)
    fig.colorbar(im, ax=eixos, shrink=0.8, label='|valor|')
    plt.show()


# Reaproveitamos o sistema de teste e guardamos as etapas do escalonamento
x_sol, etapas_gauss = eliminacao_gauss(A_teste, b_teste, pivoteamento=True, guardar_etapas=True)
plotar_etapas_eliminacao(etapas_gauss)

### Demonstração: o que acontece sem pivoteamento?

Vamos construir um sistema $2\times 2$ propositalmente "ruim": o primeiro pivô é um número muito pequeno ($\varepsilon = 10^{-20}$, bem abaixo da precisão de máquina em ponto flutuante). Sem pivoteamento, o multiplicador $m_{21} = a_{21}/\varepsilon$ fica gigantesco, e o erro de arredondamento é amplificado ao ponto de destruir a solução. Com pivoteamento parcial, a linha com o maior valor absoluto é escolhida como pivô, e o problema desaparece.

In [ ]:
# Sistema propositalmente mal-condicionado para a eliminação SEM pivoteamento:
# eps*x1 +   x2 = 1
#    x1 +   x2 = 0
# Solução exata (fazendo as contas à mão, já que eps é desprezível): x1 = -1, x2 = 1
eps = 1e-20
A_ruim = np.array([[eps, 1.0],
                    [1.0, 1.0]])
b_ruim = np.array([1.0, 0.0])
x_exato = np.array([-1.0, 1.0])  # solução de referência (eps desprezível)

x_sem_pivo = eliminacao_gauss(A_ruim, b_ruim, pivoteamento=False)
x_com_pivo = eliminacao_gauss(A_ruim, b_ruim, pivoteamento=True)

print('Pivô da coluna 1, sem troca de linhas:', eps, '(quase nulo!)')
print()
print('SEM pivoteamento -> x =', x_sem_pivo, ' | erro ||x - x_exato|| =', np.linalg.norm(x_sem_pivo - x_exato))
print('COM pivoteamento -> x =', x_com_pivo, ' | erro ||x - x_exato|| =', np.linalg.norm(x_com_pivo - x_exato))

Note o resultado: **sem pivoteamento**, o algoritmo retorna $x_1 = 0$, quando o valor correto é $x_1 = -1$ — um erro de 100%! Isso acontece porque, ao calcular $m_{21} = 1/\varepsilon = 10^{20}$ e atualizar a segunda linha, a operação $1 - m_{21}$ em ponto flutuante perde completamente a informação do "$1$" (ele é muito menor que $10^{20}$ dentro da precisão de 64 bits), e o pivô da segunda etapa já nasce corrompido. **Com pivoteamento**, a linha com o maior valor absoluto (a segunda) vira a linha-pivô, o multiplicador correspondente é $m=\varepsilon \approx 0$, e a solução sai correta. Esse é o motivo pelo qual pivoteamento parcial é praticamente obrigatório em qualquer implementação de eliminação de Gauss de uso geral.

---

## 3. Método de Gauss-Seidel

### Ideia central

Gauss-Seidel é um método **iterativo**: constrói uma sequência $\mathbf{x}^{(0)}, \mathbf{x}^{(1)}, \mathbf{x}^{(2)}, \ldots$ que converge para a solução de $A\mathbf{x}=\mathbf{b}$, sem nunca triangularizar a matriz.

Decompomos $A = D - L - U$, onde $D$ é a diagonal de $A$, $-L$ é a parte estritamente abaixo da diagonal e $-U$ é a parte estritamente acima. O sistema $A\mathbf{x}=\mathbf{b}$ se reescreve como

$$(D-L)\mathbf{x} = U\mathbf{x} + \mathbf{b}$$

o que sugere a iteração

$$\mathbf{x}^{(k+1)} = (D-L)^{-1}\left(U\mathbf{x}^{(k)} + \mathbf{b}\right)$$

Na prática, **não** calculamos essa inversa: resolvemos equação a equação, linha por linha, usando sempre os valores **mais atualizados** disponíveis:

$$x_i^{(k+1)} = \frac{1}{a_{ii}}\left(b_i - \sum_{j<i} a_{ij}\,x_j^{(k+1)} - \sum_{j>i} a_{ij}\,x_j^{(k)}\right), \qquad i = 1, \ldots, n$$

Repare que, ao calcular $x_i^{(k+1)}$, já usamos $x_1^{(k+1)}, \ldots, x_{i-1}^{(k+1)}$ (acabaram de ser atualizados nesta mesma iteração) e apenas $x_{i+1}^{(k)}, \ldots, x_n^{(k)}$ (ainda da iteração anterior). Essa é a diferença essencial em relação ao método de Jacobi (seção 6), que usa somente valores da iteração anterior.

O processo para quando o **resíduo** $\lVert \mathbf{b} - A\mathbf{x}^{(k)}\rVert$ (ou a variação $\lVert \mathbf{x}^{(k+1)} - \mathbf{x}^{(k)}\rVert$) fica menor que uma tolerância pré-definida, ou após um número máximo de iterações.

In [ ]:
def gauss_seidel(A, b, x0=None, tol=1e-10, max_iter=100, guardar_historico=True):
    """Resolve A x = b pelo método iterativo de Gauss-Seidel, implementado do zero.

    Parâmetros
    ----------
    A, b : matriz e vetor do sistema.
    x0 : estimativa inicial (padrão: vetor de zeros).
    tol : tolerância no resíduo ||b - A x|| para considerar convergência.
    max_iter : número máximo de iterações.
    guardar_historico : se True, guarda x^(k) e o resíduo a cada iteração.

    Retorna
    -------
    x : solução aproximada
    historico : lista de vetores x^(k) (incluindo x0), só se guardar_historico=True
    residuos : lista dos resíduos ||b - A x^(k)|| a cada iteração
    """
    n = len(b)
    x = np.zeros(n) if x0 is None else x0.astype(float).copy()
    historico = [x.copy()] if guardar_historico else None
    residuos = [np.linalg.norm(b - A @ x)]

    for k in range(max_iter):
        x_novo = x.copy()
        for i in range(n):
            # Soma com os valores JÁ ATUALIZADOS desta iteração (j < i)
            soma_atualizados = np.dot(A[i, :i], x_novo[:i])
            # Soma com os valores AINDA DA ITERAÇÃO ANTERIOR (j > i)
            soma_antigos = np.dot(A[i, i + 1:], x[i + 1:])
            x_novo[i] = (b[i] - soma_atualizados - soma_antigos) / A[i, i]
        x = x_novo

        if guardar_historico:
            historico.append(x.copy())
        residuo = np.linalg.norm(b - A @ x)
        residuos.append(residuo)

        if residuo < tol:
            break

    if guardar_historico:
        return x, historico, residuos
    return x, residuos


# --- Teste com um sistema 3x3 diagonalmente dominante ---
A_gs = np.array([[10.0, -1.0, 2.0],
                  [-1.0, 11.0, -1.0],
                  [2.0, -1.0, 10.0]])
b_gs = np.array([6.0, 25.0, -11.0])

x_gs, hist_gs, res_gs = gauss_seidel(A_gs, b_gs, tol=1e-12, max_iter=50)
print(f'Convergiu em {len(hist_gs) - 1} iterações')
print('Solução aproximada: x =', x_gs)
print('Resíduo final:', res_gs[-1])

### Visualizando a trajetória das iterações (sistema $2\times 2$)

Para um sistema $2\times 2$, cada equação representa uma reta no plano $(x_1, x_2)$, e a solução é o ponto de interseção das duas retas. Podemos visualizar as iterações de Gauss-Seidel como uma **trajetória em escada**: a cada meia-iteração, o método se move **horizontalmente** (atualiza $x_1$ usando a reta da equação 1) ou **verticalmente** (atualiza $x_2$ usando a reta da equação 2), de forma análoga ao *cobweb plot* usado para iteração de ponto fixo em uma dimensão.

In [ ]:
def gauss_seidel_2x2_trajetoria(A, b, x0, n_iter):
    """Versão especializada para n=2, que registra a trajetória em 'escada':
    cada atualização de UMA variável de cada vez gera um ponto no caminho,
    permitindo visualizar o movimento horizontal/vertical no plano (x1, x2).
    """
    x1, x2 = x0
    caminho = [(x1, x2)]
    for _ in range(n_iter):
        # Atualiza x1 (movimento HORIZONTAL, mantendo x2 fixo) usando a equação 1
        x1 = (b[0] - A[0, 1] * x2) / A[0, 0]
        caminho.append((x1, x2))
        # Atualiza x2 (movimento VERTICAL, mantendo x1 fixo) usando a equação 2
        x2 = (b[1] - A[1, 0] * x1) / A[1, 1]
        caminho.append((x1, x2))
    return np.array(caminho)


# Sistema 2x2 diagonalmente dominante:
# 4 x1 +   x2 = 9
#   x1 + 3 x2 = 10
A_2d = np.array([[4.0, 1.0], [1.0, 3.0]])
b_2d = np.array([9.0, 10.0])
solucao_2d = eliminacao_gauss(A_2d, b_2d)  # usada só como referência p/ plot

caminho = gauss_seidel_2x2_trajetoria(A_2d, b_2d, x0=(0.0, 0.0), n_iter=5)

fig, ax = plt.subplots(figsize=(7, 7))

# Retas das duas equações: isolamos x2 em função de x1 em cada uma
x1_vals = np.linspace(-1, 4, 200)
reta1 = b_2d[0] / A_2d[0, 1] - (A_2d[0, 0] / A_2d[0, 1]) * x1_vals   # da equação 1
reta2 = (b_2d[1] - A_2d[1, 0] * x1_vals) / A_2d[1, 1]                 # da equação 2
ax.plot(x1_vals, reta1, color=COR_GAUSS, lw=2, label=r'Eq. 1: $4x_1+x_2=9$')
ax.plot(x1_vals, reta2, color=COR_DIVERGE, lw=2, label=r'Eq. 2: $x_1+3x_2=10$')

# Trajetória em escada das iterações
ax.plot(caminho[:, 0], caminho[:, 1], color=COR_SEIDEL, marker='o', markersize=4,
        linewidth=1.2, linestyle='--', label='Trajetória de Gauss-Seidel')
ax.scatter(*caminho[0], color='black', zorder=5, label='Início $x^{(0)}$')
ax.scatter(*solucao_2d, color=COR_DESTAQUE, marker='*', s=250, zorder=5,
           label='Solução exata')

ax.set_xlabel('$x_1$')
ax.set_ylabel('$x_2$')
ax.set_title('Trajetória de Gauss-Seidel no espaço de soluções (estilo "cobweb")')
ax.legend(loc='upper right')
ax.set_xlim(-0.5, 3.5)
ax.set_ylim(0, 4.5)
plt.show()

In [ ]:
# Gráfico de convergência: resíduo ||b - A x^(k)|| por iteração, em escala log.
# Uma reta decrescente em escala log indica convergência geométrica (linear).
plt.figure()
plt.semilogy(range(len(res_gs)), res_gs, color=COR_SEIDEL, marker='o', markersize=4,
             label='Gauss-Seidel (sistema 3x3 diag. dominante)')
plt.xlabel('Iteração $k$')
plt.ylabel(r'Resíduo $\|b - Ax^{(k)}\|$ (escala log)')
plt.title('Convergência do método de Gauss-Seidel')
plt.legend()
plt.show()

---

## 4. Convergência de Gauss-Seidel

Diferentemente da Eliminação de Gauss, que sempre funciona (desde que $A$ seja não singular e usemos pivoteamento), o método de Gauss-Seidel **nem sempre converge**. Precisamos entender quando podemos confiar nele.

### Condição suficiente: dominância diagonal

Uma matriz $A$ é **diagonalmente dominante** (em linhas, no sentido estrito) se, para toda linha $i$:

$$|a_{ii}| > \sum_{j \neq i} |a_{ij}|$$

ou seja, o elemento da diagonal é, em módulo, maior que a soma dos demais elementos da mesma linha. **Se $A$ é diagonalmente dominante, Gauss-Seidel (e também Jacobi) converge para a solução exata, para qualquer estimativa inicial $\mathbf{x}^{(0)}$.** Essa é uma condição *suficiente*, não necessária — existem matrizes não diagonalmente dominantes para as quais o método ainda converge.

### Condição necessária e suficiente: raio espectral

Reescrevendo a iteração de Gauss-Seidel na forma $\mathbf{x}^{(k+1)} = T\,\mathbf{x}^{(k)} + \mathbf{c}$, com a **matriz de iteração**

$$T = (D - L)^{-1} U$$

o resultado teórico exato é: **o método converge, para qualquer $\mathbf{x}^{(0)}$, se e somente se o raio espectral de $T$ for menor que 1**:

$$\rho(T) = \max_i |\lambda_i(T)| < 1$$

onde $\lambda_i(T)$ são os autovalores de $T$. Quanto **menor** o raio espectral, **mais rápida** é a convergência — o erro decai aproximadamente como $\rho(T)^k$ a cada iteração $k$. Usaremos `numpy.linalg.eigvals` **apenas para calcular esses autovalores com fins de análise/diagnóstico** (não para resolver o sistema em si — isso continua sendo feito pelo nosso `gauss_seidel` implementado do zero).

### Demonstração numérica: um caso que converge e um caso que diverge

Vamos comparar dois sistemas $3\times 3$:

- **Caso A (converge rápido)**: matriz diagonalmente dominante.
- **Caso B (diverge/oscila)**: mesma estrutura, mas com os elementos fora da diagonal "invertidos" em magnitude, violando a dominância diagonal.

In [ ]:
def eh_diagonalmente_dominante(A):
    """Verifica se A é diagonalmente dominante (estrito, por linhas)."""
    n = A.shape[0]
    for i in range(n):
        soma_fora_diag = np.sum(np.abs(A[i, :])) - np.abs(A[i, i])
        if np.abs(A[i, i]) <= soma_fora_diag:
            return False
    return True


def raio_espectral_gauss_seidel(A):
    """Calcula o raio espectral da matriz de iteração T = (D-L)^-1 U de Gauss-Seidel.

    IMPORTANTE: isto usa numpy.linalg apenas para fins de ANÁLISE/diagnóstico
    teórico (calcular autovalores de T). A resolução do sistema em si continua
    sendo feita pelo nosso `gauss_seidel`, implementado do zero.
    """
    D = np.diag(np.diag(A))
    L = -np.tril(A, -1)
    U = -np.triu(A, 1)
    T = np.linalg.inv(D - L) @ U
    autovalores = np.linalg.eigvals(T)
    return np.max(np.abs(autovalores))


# --- Caso A: diagonalmente dominante (o mesmo sistema 3x3 já usado acima) ---
A_caso_a = A_gs
b_caso_a = b_gs

# --- Caso B: mesma 'forma', mas com a diagonal fraca e o resto forte ---
# (violando propositalmente a dominância diagonal)
A_caso_b = np.array([[1.0, 2.0, 3.0],
                      [2.0, 1.0, 2.0],
                      [3.0, 2.0, 1.0]])
b_caso_b = np.array([6.0, 5.0, 6.0])  # solução exata é (1, 1, 1)

for nome, A, b in [('Caso A (dominante)', A_caso_a, b_caso_a),
                    ('Caso B (não dominante)', A_caso_b, b_caso_b)]:
    dominante = eh_diagonalmente_dominante(A)
    rho = raio_espectral_gauss_seidel(A)
    print(f'{nome}: diagonalmente dominante = {dominante}, raio espectral rho(T) = {rho:.4f}')

In [ ]:
# Rodamos Gauss-Seidel (nossa implementação do zero) nos dois casos e comparamos
# a evolução do resíduo, lado a lado, em escala log.
N_ITER_DEMO = 15
_, _, res_caso_a = gauss_seidel(A_caso_a, b_caso_a, tol=1e-14, max_iter=N_ITER_DEMO)
_, _, res_caso_b = gauss_seidel(A_caso_b, b_caso_b, tol=1e-14, max_iter=N_ITER_DEMO)

rho_a = raio_espectral_gauss_seidel(A_caso_a)
rho_b = raio_espectral_gauss_seidel(A_caso_b)

plt.figure()
plt.semilogy(range(len(res_caso_a)), res_caso_a, color=COR_SEIDEL, marker='o',
             label=f'Caso A: dominante, $\\rho(T)={rho_a:.3f}$ -> converge')
plt.semilogy(range(len(res_caso_b)), res_caso_b, color=COR_DIVERGE, marker='s',
             label=f'Caso B: não dominante, $\\rho(T)={rho_b:.3f}$ -> diverge')
plt.xlabel('Iteração $k$')
plt.ylabel(r'Resíduo $\|b - Ax^{(k)}\|$ (escala log)')
plt.title('Convergência x Divergência: efeito da dominância diagonal / raio espectral')
plt.legend()
plt.show()

print(f'Caso A: rho(T) = {rho_a:.4f} < 1  => convergência garantida (e rápida, pois rho é pequeno)')
print(f'Caso B: rho(T) = {rho_b:.4f} > 1  => divergência garantida, não importa o x0 escolhido')

O contraste é nítido: no **Caso A**, com $\rho(T) \approx 0.043 \ll 1$, o resíduo cai quase 2 ordens de grandeza por iteração — convergência rápida. No **Caso B**, $\rho(T) \approx 3.46 > 1$, e o resíduo **cresce geometricamente** a cada iteração — o método diverge de forma garantida, por pior que seja a sorte (ou a escolha) do chute inicial $\mathbf{x}^{(0)}$. Isso confirma, na prática, a teoria: o raio espectral da matriz de iteração é o verdadeiro critério (necessário e suficiente) de convergência, e a dominância diagonal é apenas uma condição suficiente mais fácil de verificar na mão.

---

## 5. Comparação final: Gauss vs. Gauss-Seidel

Agora vamos resolver o **mesmo sistema** $10\times 10$, bem condicionado e diagonalmente dominante, pelos dois métodos, e comparar:

- número de operações (determinístico, $O(n^3)$, para Gauss) vs. número de iterações (até a tolerância, para Gauss-Seidel);
- tempo de execução (apenas ilustrativo — para $n=10$ a diferença é irrelevante, mas a tendência assintótica é o que importa);
- precisão final (resíduo e erro em relação à solução exata conhecida).

Para ter uma solução exata de referência, construímos o sistema de trás para frente: escolhemos $\mathbf{x}_{\text{real}} = (1, 2, \ldots, 10)$ e calculamos $\mathbf{b} = A\,\mathbf{x}_{\text{real}}$, garantindo que sabemos exatamente qual deveria ser a resposta.

In [ ]:
import time

np.random.seed(42)  # reprodutibilidade
n = 10

# Geramos uma matriz aleatória e depois reforçamos a diagonal para garantir
# dominância diagonal estrita (e, portanto, boa condição + convergência).
A_comp = np.random.uniform(-2, 2, size=(n, n))
for i in range(n):
    A_comp[i, i] = np.sum(np.abs(A_comp[i, :])) + np.random.uniform(1, 3)

x_real = np.arange(1, n + 1, dtype=float)   # solução conhecida: (1, 2, ..., 10)
b_comp = A_comp @ x_real

print('Diagonalmente dominante?', eh_diagonalmente_dominante(A_comp))
print('Raio espectral de Gauss-Seidel:', f'{raio_espectral_gauss_seidel(A_comp):.4f}')

# --- Eliminação de Gauss ---
t0 = time.perf_counter()
x_gauss_comp = eliminacao_gauss(A_comp, b_comp)
t_gauss = time.perf_counter() - t0

# --- Gauss-Seidel ---
t0 = time.perf_counter()
x_seidel_comp, hist_comp, res_comp = gauss_seidel(A_comp, b_comp, tol=1e-12, max_iter=200)
t_seidel = time.perf_counter() - t0

n_iter_gauss = (2 * n**3) // 3   # ordem de grandeza de operações de ponto flutuante
n_iter_seidel = len(hist_comp) - 1

print()
print(f'{"Método":<20}{"Operações/Iterações":<22}{"Tempo (s)":<14}{"Erro vs. solução exata":<25}')
print(f'{"Eliminação de Gauss":<20}{f"~{n_iter_gauss} FLOPs (O(n^3))":<22}{t_gauss:<14.6f}{np.linalg.norm(x_gauss_comp - x_real):<25.3e}')
print(f'{"Gauss-Seidel":<20}{f"{n_iter_seidel} iterações":<22}{t_seidel:<14.6f}{np.linalg.norm(x_seidel_comp - x_real):<25.3e}')

In [ ]:
# Visualizamos a convergência de Gauss-Seidel neste sistema 10x10, e marcamos
# o resíduo final da Eliminação de Gauss como referência (reta horizontal),
# já que esta não é iterativa.
plt.figure()
plt.semilogy(range(len(res_comp)), res_comp, color=COR_SEIDEL, marker='o', markersize=4,
             label='Gauss-Seidel (resíduo por iteração)')
residuo_gauss_comp = np.linalg.norm(b_comp - A_comp @ x_gauss_comp)
plt.axhline(residuo_gauss_comp, color=COR_GAUSS, linestyle='--',
            label=f'Eliminação de Gauss (resíduo final, direto): {residuo_gauss_comp:.1e}')
plt.xlabel('Iteração $k$')
plt.ylabel(r'Resíduo $\|b - Ax\|$ (escala log)')
plt.title('Sistema 10x10: convergência de Gauss-Seidel vs. solução direta de Gauss')
plt.legend()
plt.show()

**Discussão:** para este sistema $10\times 10$ bem condicionado, ambos os métodos atingem erro da ordem de $10^{-14}$–$10^{-15}$ (precisão de máquina), e o tempo de execução de ambos é desprezível (frações de milissegundo). A Eliminação de Gauss precisa de um número fixo de operações ($O(n^3)$, independente da matriz específica), enquanto Gauss-Seidel precisou de pouco mais de uma dezena de iterações para convergir — cada iteração custando apenas $O(n^2)$ (um produto matriz-vetor).

Para $n=10$ essa diferença não importa na prática. Mas **conceitualmente** ela é decisiva: se $n$ fosse $10^5$ (uma malha de elementos finitos de um casco, por exemplo) e a matriz fosse esparsa, a Eliminação de Gauss (que não tira proveito da esparsidade da mesma forma, e tem custo $O(n^3)$) se tornaria inviável, enquanto Gauss-Seidel, com custo por iteração proporcional ao número de elementos não nulos, continuaria praticável — desde que a matriz seja diagonalmente dominante (ou, de forma mais geral, $\rho(T) < 1$) para garantir convergência em poucas iterações.

---

## 6. Para ir além: Jacobi vs. Gauss-Seidel

O método de **Jacobi** é o "primo" mais simples de Gauss-Seidel. A fórmula de atualização é quase idêntica:

$$x_i^{(k+1)} = \frac{1}{a_{ii}}\left(b_i - \sum_{j \neq i} a_{ij}\,x_j^{(k)}\right)$$

A diferença crucial: Jacobi usa **somente** valores da iteração anterior $\mathbf{x}^{(k)}$ para calcular todo o vetor $\mathbf{x}^{(k+1)}$ (as atualizações podem, inclusive, ser feitas em paralelo, todas ao mesmo tempo). Gauss-Seidel, por sua vez, já aproveita, dentro da própria iteração, os componentes de $\mathbf{x}^{(k+1)}$ que acabaram de ser calculados — ou seja, usa informação "mais fresca" assim que ela fica disponível. Por isso, **Gauss-Seidel geralmente converge mais rápido que Jacobi** para o mesmo sistema (embora, por depender de atualizações sequenciais, seja menos trivial de paralelizar).

Vamos comparar os dois no mesmo sistema $10\times 10$ da seção anterior.

In [ ]:
def jacobi(A, b, x0=None, tol=1e-10, max_iter=200):
    """Resolve A x = b pelo método de Jacobi, implementado do zero.

    Diferença em relação a Gauss-Seidel: TODAS as atualizações usam somente
    os valores x^(k) da iteração anterior (nunca os valores já recalculados
    na mesma iteração), por isso usamos um vetor x_novo totalmente separado.
    """
    n = len(b)
    x = np.zeros(n) if x0 is None else x0.astype(float).copy()
    residuos = [np.linalg.norm(b - A @ x)]

    for k in range(max_iter):
        x_novo = np.zeros(n)
        for i in range(n):
            # Soma de TODOS os termos fora da diagonal, usando sempre x (antigo)
            soma = np.dot(A[i, :], x) - A[i, i] * x[i]
            x_novo[i] = (b[i] - soma) / A[i, i]
        x = x_novo
        residuo = np.linalg.norm(b - A @ x)
        residuos.append(residuo)
        if residuo < tol:
            break

    return x, residuos


x_jacobi_comp, res_jacobi_comp = jacobi(A_comp, b_comp, tol=1e-12, max_iter=200)

print(f'Jacobi:       {len(res_jacobi_comp) - 1} iterações, erro = {np.linalg.norm(x_jacobi_comp - x_real):.3e}')
print(f'Gauss-Seidel: {len(hist_comp) - 1} iterações, erro = {np.linalg.norm(x_seidel_comp - x_real):.3e}')

plt.figure()
plt.semilogy(range(len(res_jacobi_comp)), res_jacobi_comp, color=COR_JACOBI, marker='^',
             markersize=4, label=f'Jacobi ({len(res_jacobi_comp) - 1} iterações)')
plt.semilogy(range(len(res_comp)), res_comp, color=COR_SEIDEL, marker='o',
             markersize=4, label=f'Gauss-Seidel ({len(hist_comp) - 1} iterações)')
plt.xlabel('Iteração $k$')
plt.ylabel(r'Resíduo $\|b - Ax^{(k)}\|$ (escala log)')
plt.title('Jacobi vs. Gauss-Seidel no mesmo sistema 10x10')
plt.legend()
plt.show()

Como esperado, Jacobi precisou de mais iterações que Gauss-Seidel para atingir a mesma tolerância, confirmando que usar informação "atualizada na hora" acelera a convergência. Em problemas de engenharia naval com matrizes grandes e esparsas (por exemplo, a resolução iterativa de uma malha de diferenças finitas para o escoamento potencial em torno de um casco), essa diferença de velocidade de convergência pode representar uma redução significativa no tempo total de simulação.

---

## Resumo

| | Eliminação de Gauss | Gauss-Seidel |
|---|---|---|
| Tipo | Método direto | Método iterativo |
| Custo | $O(n^3)$, fixo | $O(n^2)$ por iteração (ou $O(\text{nnz})$ se esparsa) |
| Garantia de solução | Sempre (se $A$ não singular + pivoteamento) | Só garantida se $\rho(T) < 1$ (ex.: dominância diagonal) |
| Melhor cenário | $n$ pequeno/moderado, matriz densa | $n$ grande, matriz esparsa e bem condicionada |

Entender quando usar cada abordagem — e como garantir que um método iterativo de fato converge — é uma habilidade essencial para qualquer engenheiro que lide com simulações numéricas de grande porte, como as encontradas em projeto estrutural e hidrodinâmico de embarcações.